# 19 — Open-data audit and frozen regional equity datasets

Build **Poland, Czechia, Romania and Sweden**, each for **unhedged USD and EUR investors**.
The fixed observation endpoint is **September 2026**. New national observations are downloaded on
**16 November 2026**; the common global inputs retain Hungary's original **13 November 2026**
vintage. These are separate vintages, not a reconstructed historical real-time information set.
Use the existing **base** kernel. No API key, paid database, ETF substitution or new package is needed.

Keep each country's available sample and export a **matched common sample including Hungary**.
Use total-return indices, never extend them with a price-only index, interpolate returns or fill
missing market/FX observations. This notebook prepares inputs; it does not estimate pricing models
or compare hedging strategies. Existing Hungarian data and research outputs remain unchanged.


In [1]:
START_MONTH = "2000-01"
END_MONTH = "2026-09"       # Fixed research endpoint; not today's latest observation.
SNAPSHOT_DATE = "2026-11-16"

from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
from io import BytesIO, StringIO
from html.parser import HTMLParser
import hashlib, html, importlib.metadata, json, os, platform, re, sys
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display, Markdown

ROOT = Path.cwd()
LOCK = json.loads((ROOT / "research_synthesis_inputs.json").read_text())
HU_ROOT = ROOT / LOCK["snapshot"]
RUN = ROOT / f"data/regional_fixed_{SNAPSHOT_DATE}_end_{END_MONTH}/from_{START_MONTH}"
RAW = RUN / "raw"
MANIFEST = RUN / "manifest.json"
first, last = pd.Period(START_MONTH, "M"), pd.Period(END_MONTH, "M")
assert first >= pd.Period("2000-01", "M") and first <= last <= pd.Period("2026-09", "M")
settings = {"recipe_version": 1, "requested_start": START_MONTH, "endpoint": END_MONTH,
            "national_download_date": SNAPSHOT_DATE, "global_vintage": "2026-11-13"}
if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text())
    assert manifest["settings"] == settings, "Use a separate snapshot folder for different settings."
else:
    assert str(datetime.now(ZoneInfo("Europe/Vienna")).date()) == SNAPSHOT_DATE, "Choose a new snapshot date; a later download cannot recreate this vintage."
    manifest = {"settings": settings, "sealed": False, "sources": {}, "inherited": {}, "exports": {}}
RAW.mkdir(parents=True, exist_ok=True)

def digest(raw): return hashlib.sha256(raw).hexdigest()
def save_manifest():
    temp = MANIFEST.with_suffix(".tmp")
    temp.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    temp.replace(MANIFEST)
def inherited(path):
    relative = path.relative_to(ROOT).as_posix()
    raw = path.read_bytes()
    if digest(raw) != LOCK["files"][relative]: raise ValueError(f"Hungarian input changed: {relative}")
    if manifest["sealed"]:
        assert manifest["inherited"][relative] == digest(raw)
    else: manifest["inherited"][relative] = digest(raw)
    return raw

session = requests.Session()
session.headers["User-Agent"] = "curl/8.7.1"
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET", "POST"])))

def fetch(name, url, *, params=None, post=None, client=None, details=None):
    path = RAW / name
    if path.exists():
        raw = path.read_bytes()
        assert name in manifest["sources"] and digest(raw) == manifest["sources"][name]["sha256"], f"Corrupt/unrecorded source: {name}"
        return raw
    if manifest["sealed"]: raise FileNotFoundError(f"Restore frozen source {path}; no automatic re-fetch is allowed.")
    assert str(datetime.now(ZoneInfo("Europe/Vienna")).date()) == SNAPSHOT_DATE, "Download date changed; start a new vintage."
    response = (client or session).request("POST" if post is not None else "GET", url,
        params=params, data=post, timeout=(10, 45))
    response.raise_for_status()
    raw = response.content
    assert raw, f"Empty source: {name}"
    path.write_bytes(raw)
    manifest["sources"][name] = {"url": response.url, "method": response.request.method,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(), "sha256": digest(raw),
        "bytes": len(raw), "details": details or {}}
    save_manifest()
    return raw

print("Python:", sys.executable)
print("Packages:", {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "requests", "ipykernel"]})
print("Snapshot:", RUN.relative_to(ROOT), "| sealed:", manifest["sealed"])


Python: /Users/lajosgalambos/miniconda3/bin/python
Packages: {'numpy': '2.4.4', 'pandas': '3.0.2', 'requests': '2.32.5', 'ipykernel': '7.2.0'}
Snapshot: data/regional_fixed_2026-11-16_end_2026-09/from_2000-01 | sealed: False


## Source audit and comparability

| Country | Equity input | Return convention and coverage limitation |
|---|---|---|
| Poland | [GPW Benchmark WIG](https://gpwbenchmark.pl/karta-indeksu?isin=PL9999999995) | Broad listed-market total return, including dividends and subscription rights; available for our full requested period |
| Czechia | [PSE PX-TR](https://www.pse.cz/en/indices/index-values/detail/CZ0160000019) and [official CSV](https://ftp.pse.cz/Info.bas/Cz/PX-TR.csv) | Gross dividends; history begins March 2006; launched March 2014 with back-calculated earlier values |
| Romania | [BVB BET-TR](https://www.bvb.ro/FinancialInstruments/Indices/IndicesProfiles.aspx?i=BET-TR) | Gross dividends; documented back-history begins September 2012, but this public closing feed may start later; actual downloadable coverage is measured below |
| Sweden | [Nasdaq OMXSGI](https://indexes.nasdaqomx.com/Index/Overview/OMXSGI), distributed by [FRED](https://fred.stlouisfed.org/series/NASDAQOMXSGI) | All-share gross return in SEK; this open feed begins November 2008, even though the index has an earlier base date |

Methodology evidence: [WIG](https://gpwbenchmark.pl/pub/BENCHMARK/files/PDF/metodologia_indeksow/new/2022_12_30_WIG_en.pdf),
[PSE 2014 report, p. 5](https://ftp.pse.cz/Info.bas/Cz/vz14-cz-PSE.pdf),
[BET-TR rules](https://bvb.ro/info/indices/2025/BVB-EN_Manual%20BET-TR_V_01_2022.pdf).
The return coverage audit below measures the **downloaded data**, rather than assuming an index's
launch/base date guarantees an accessible history. First partial index months only supply a level
for the next month's return. Back-calculated observations are flagged, not silently treated as live.

New national FX comes from [ECB reference rates](https://data.ecb.europa.eu/help/api/data-examples),
national all-items HICP from [Eurostat](https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_minr/default/table?lang=en).
World/US portfolios, USD/EUR cash proxies, US predictors, EUR/USD and US/euro-area inflation are
reused from the locked Hungarian snapshot. EUR cash remains the same Euribor proxy.

These are different listed-market portfolios, not sector-matched investments. Dividend-tax,
constituent eligibility, concentration and fixing-time differences limit later economic comparisons.
Public access is not an unrestricted redistribution licence; raw data remain in the ignored `data/` folder.


In [2]:
COUNTRIES = {
    "PL": {"country": "Poland", "currency": "PLN", "index": "WIG", "known_backcast_before": None},
    "CZ": {"country": "Czechia", "currency": "CZK", "index": "PX-TR", "known_backcast_before": "2014-04"},
    "RO": {"country": "Romania", "currency": "RON", "index": "BET-TR", "known_backcast_before": "2014-10"},
    "SE": {"country": "Sweden", "currency": "SEK", "index": "OMXSGI", "known_backcast_before": None},
}
equity_raw = {}
equity_raw["PL"] = fetch("poland_wig.json", "https://gpwbenchmark.pl/chart-json.php", params={
    "req": json.dumps([{"isin": "PL9999999995", "mode": "RANGE", "from": "1999-12-01",
                        "to": str(last.end_time.date())}], separators=(",", ":"))})
equity_raw["CZ"] = fetch("czech_pxtr.csv", "https://ftp.pse.cz/Info.bas/Cz/PX-TR.csv")
equity_raw["SE"] = fetch("sweden_omxsgi.csv", "https://fred.stlouisfed.org/graph/fredgraph.csv", params={
    "id": "NASDAQOMXSGI", "cosd": "1998-01-01", "coed": str(last.end_time.date())})
fx_raw = fetch("ecb_country_fx.csv", "https://data-api.ecb.europa.eu/service/data/EXR/D.PLN+CZK+RON+SEK.EUR.SP00.A",
    params={"startPeriod": "1998-01-01", "endPeriod": str(last.end_time.date()), "format": "csvdata"})
hicp_raw = fetch("national_hicp.json", "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_minr",
    params={"lang": "en", "freq": "M", "coicop18": "TOTAL", "unit": "I25",
            "geo": list(COUNTRIES), "sinceTimePeriod": "1998-01"})
print("Downloaded/verified Poland, Czechia, Sweden, national FX and national HICP.")


Downloaded/verified Poland, Czechia, Sweden, national FX and national HICP.


## Romanian public closing-value history

BVB's quick-download link only supports its five most recent sessions; it cannot provide our
historical endpoint. Its chart API also failed the access audit. Use the ordinary public
**Closing values → date selector**, with its standard asynchronous form request and browser-compatible
rendering. No account, personal API key or chart token is used. Save one response for each requested
month, check its actual dates and index header, and reuse those files offline after sealing.

**Audit finding:** the closing feed returns explicit `N/A` values before September 2014 despite the
documented 2012 back-history. Those gaps remain missing. The usable Romanian monthly returns start
in October 2014, which determines the matched window; no back-history is invented from the base date.


In [3]:
class HiddenFields(HTMLParser):
    def __init__(self, text):
        super().__init__(); self.values = {}; self.feed(text)
    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "input" and attrs.get("type") == "hidden" and "name" in attrs:
            self.values[attrs["name"]] = attrs.get("value", "")

def romania_rows(raw):
    text = raw.decode("utf-8-sig")
    table = re.search(r'<table\b[^>]*id="gvCV"[^>]*>(.*?)</table>', text, re.S)
    assert table and "BET-TR (RON)" in table[0], "BVB returned no BET-TR closing table."
    records = []
    for row in re.findall(r"<tr\b[^>]*>(.*?)</tr>", table[1], re.S):
        cells = [html.unescape(re.sub(r"<[^>]+>", "", c)).strip()
                 for c in re.findall(r"<t[dh]\b[^>]*>(.*?)</t[dh]>", row, re.S)]
        if cells and re.fullmatch(r"\d{2}\.\d{2}\.\d{4}", cells[0]):
            assert len(cells) == 10
            value = np.nan if cells[1] == "N/A" else float(cells[1].replace(".", "").replace(",", "."))
            records.append((pd.to_datetime(cells[0], format="%d.%m.%Y"), value))
    assert records, "BVB closing table is empty."
    return pd.DataFrame(records, columns=["date", "level"])

ro_months = pd.period_range("2012-09", last, freq="M")
assert last >= ro_months[0]
ro_names = [f"romania_bettr_{month}.html" for month in ro_months]
need_ro = any(not (RAW / name).exists() for name in ro_names)
if need_ro and manifest["sealed"]: raise FileNotFoundError("Restore missing frozen Romanian responses.")
if need_ro:
    bvb = requests.Session()
    bvb.headers["User-Agent"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    bvb.mount("https://", session.get_adapter("https://"))
    ro_url = "https://www.bvb.ro/FinancialInstruments/Indices/IndicesProfiles.aspx?i=BET-TR"
    page = bvb.get(ro_url, timeout=(10, 45)); page.raise_for_status()
    prefix = "ctl00$ctl00$body$rightColumnPlaceHolder$"
    selection = {prefix + "IndexProfilesCurrentValues$IndexControlList$ddIndices": "BET-TR"}
    form = {**HiddenFields(page.text).values, **selection,
            "__EVENTTARGET": prefix + "TabsControl$lb3", "__EVENTARGUMENT": ""}
    page = bvb.post(ro_url, data=form, timeout=(10, 45)); page.raise_for_status()
    form = {**HiddenFields(page.text).values, **selection, "__EVENTTARGET": "txtDateI",
            "__EVENTARGUMENT": "", "__ASYNCPOST": "true",
            "ctl00$ctl00$MasterScriptManager": prefix + "IndexClosingValues$PDC$upddate|txtDateI"}
    bvb.headers.update({"X-MicrosoftAjax": "Delta=true", "X-Requested-With": "XMLHttpRequest"})

ro_parts = []
for month, name in zip(ro_months, ro_names):
    if (RAW / name).exists():
        raw = fetch(name, "https://www.bvb.ro/FinancialInstruments/Indices/IndicesProfiles.aspx?i=BET-TR")
    else:
        date_text = month.end_time.strftime("%d.%m.%Y")
        payload = {**form, "txtDateI": date_text,
                   prefix + "IndexClosingValues$PDC$txtDateI": date_text}
        raw = fetch(name, ro_url, post=payload, client=bvb,
                    details={"index": "BET-TR", "selected_end_date": date_text,
                             "public_form": "Closing values, date selector"})
    part = romania_rows(raw)
    part = part.loc[part.date.dt.to_period("M").eq(month)]
    assert not part.empty and (month.end_time.normalize() - part.date.max()).days <= 7, f"BVB returned wrong/stale calendar month: {month}"
    ro_parts.append(part)
    if month.month == 12 or month == last: print("Romania verified through", month, flush=True)
ro_daily = pd.concat(ro_parts).sort_values("date")
assert not ro_daily.date.duplicated().any()
print("Romania:", int(ro_daily.level.notna().sum()), "numeric daily levels;", int(ro_daily.level.isna().sum()), "explicit N/A entries;", len(ro_months), "calendar months audited.")


Romania verified through 2012-12


Romania verified through 2013-12


Romania verified through 2014-12


Romania verified through 2015-12


Romania verified through 2016-12


Romania verified through 2017-12


... (7 outputs left out)

Romania verified through

 2025-12


Romania verified through 2026-09


Romania: 2970 numeric daily levels; 506 explicit N/A entries; 169 calendar months audited.


## Calendar alignment and inherited global inputs

GPW timestamps represent **Warsaw local midnight**; converting them to Warsaw dates before
monthly aggregation prevents previous-day/month errors. Keep each series' last valid observation
within its calendar month and report the actual dates. Index closes and ECB reference fixings are
not simultaneous. FRED's Nasdaq series also has reporting-calendar gaps; no daily values are filled.

Reuse the global returns, cash proxies and financial predictors exactly. Preserve the raw US CPI
gap in October 2025; only its **annual inflation rate** is carried for at most one month for
conditioning variables, with flags. National inflation is HICP, while US inflation is CPI.


In [4]:
def eurostat_frame(raw):
    j = json.loads(raw)
    assert j["id"] == ["freq", "unit", "coicop18", "geo", "time"] and j["size"][:3] == [1, 1, 1]
    values = np.full(int(np.prod(j["size"])), np.nan)
    for position, value in j["value"].items(): values[int(position)] = value
    values = values.reshape(j["size"])[0, 0, 0]
    geo = j["dimension"]["geo"]["category"]["index"]
    time = j["dimension"]["time"]["category"]["index"]
    dates = sorted(time, key=time.get)
    return pd.DataFrame({g: values[i] for g, i in geo.items()}, index=pd.PeriodIndex(dates, freq="M"))

def month_end(daily):
    daily = daily.sort_values("date").loc[lambda d: d.date <= last.end_time].dropna(subset=["level"])
    assert not daily.date.duplicated().any() and np.isfinite(daily.level).all() and daily.level.gt(0).all()
    daily = daily.assign(month=daily.date.dt.to_period("M"))
    rows = daily.groupby("month", sort=True).tail(1).set_index("month")
    rows.index.name = "month"
    return rows[["level", "date"]]

gpw = json.loads(equity_raw["PL"])
assert len(gpw) == 1 and gpw[0]["isin"] == "PL9999999995"
pl = pd.DataFrame(gpw[0]["data"])
pl = pd.DataFrame({"date": pd.to_datetime(pl.t, unit="s", utc=True).dt.tz_convert("Europe/Warsaw").dt.tz_localize(None).dt.normalize(), "level": pl.c})
cz = pd.read_csv(BytesIO(equity_raw["CZ"]), header=None, names=["date", "level", "change_pct"])
cz["date"] = pd.to_datetime(cz.date, format="%d.%m.%Y")
se = pd.read_csv(BytesIO(equity_raw["SE"]), na_values=".").dropna(subset=["NASDAQOMXSGI"])
se = pd.DataFrame({"date": pd.to_datetime(se.observation_date), "level": se.NASDAQOMXSGI})
levels = {c: month_end(d) for c, d in {"PL": pl, "CZ": cz, "RO": ro_daily, "SE": se}.items()}

grid = pd.period_range("1998-01", last, freq="M")
national_hicp = eurostat_frame(hicp_raw).reindex(grid)
fx_daily = pd.read_csv(BytesIO(fx_raw), usecols=["TIME_PERIOD", "CURRENCY", "OBS_VALUE"])
assert not fx_daily.duplicated(["TIME_PERIOD", "CURRENCY"]).any()
fx_daily["TIME_PERIOD"] = pd.to_datetime(fx_daily.TIME_PERIOD)
national_fx = {}
fx_dates = {}
for country, info in COUNTRIES.items():
    part = fx_daily.loc[fx_daily.CURRENCY.eq(info["currency"])].rename(columns={"TIME_PERIOD": "date", "OBS_VALUE": "level"})
    monthly = month_end(part)
    national_fx[country] = monthly.level.reindex(grid)
    fx_dates[country] = monthly.date

hu = pd.read_csv(BytesIO(inherited(HU_ROOT / "dataset.csv")), index_col=0, float_precision="round_trip")
hu.index = pd.PeriodIndex(hu.index, freq="M")
hu_inputs = pd.read_csv(BytesIO(inherited(HU_ROOT / "monthly_inputs.csv")), index_col=0, float_precision="round_trip")
hu_inputs.index = pd.PeriodIndex(hu_inputs.index, freq="M")
fred = pd.read_csv(BytesIO(inherited(HU_ROOT / "raw/fred.csv")), index_col=0, parse_dates=True, na_values=".")
fred = fred.resample("ME").last().to_period("M").reindex(grid)
ea_hicp = eurostat_frame(inherited(HU_ROOT / "raw/eurostat_hicp.json"))["EA"].reindex(grid)
inflation_us_raw = fred.CPIAUCNS.pct_change(12, fill_method=None) * 100
inflation_us = inflation_us_raw.ffill(limit=1)
inflation_ea = ea_hicp.pct_change(12, fill_method=None) * 100
us_carried = inflation_us_raw.isna() & inflation_us.notna()
old_fx = pd.read_csv(BytesIO(inherited(HU_ROOT / "raw/ecb_fx.csv")), usecols=["TIME_PERIOD", "CURRENCY", "OBS_VALUE"])
old_usd = old_fx.loc[old_fx.CURRENCY.eq("USD")].rename(columns={"TIME_PERIOD": "date", "OBS_VALUE": "level"})
old_usd["date"] = pd.to_datetime(old_usd.date)
usd_per_eur = month_end(old_usd).level.reindex(grid)  # Includes Dec 1999 for the Jan 2000 conversion.
eur_usd = hu.fx_eur_usd.reindex(grid)
global_columns = [c for c in hu if c.startswith(("r_world_", "r_us_", "rx_world_", "rx_us_", "rf_")) or c in [
    "fx_eur_usd", "short_rate_change_lag1_pp", "term_spread_change_lag1_pp", "credit_spread_lag1_pp", "flag_us_inflation_carried_lag2"]]
print("Verified locked global inputs; downloaded equity endpoints:")
display(pd.DataFrame({c: {"first_level": str(d.index.min()), "last_level": str(d.index.max()), "months": len(d)} for c, d in levels.items()}).T)


Verified locked global inputs; downloaded equity endpoints:


,first_level,last_level,months
PL,1999-12,2026-09,322
CZ,2006-03,2026-09,247
RO,2014-09,2026-09,145
SE,2008-11,2026-09,215


## Country conditioning information

Audit these **candidate** regime indicators rather than copying Hungary's March 2008 dummy:
Poland's float starts on [12 April 2000](https://www.knf.gov.pl/en/ABOUT_US/The_Polish_Financial_Supervision_Authority_UKNF/UKNF_Supervisory_Blog?articleId=87534&p_id=19)
(flag from May, the next full month); the [Czech exchange-rate floor](https://www.cnb.cz/en/monetary-policy/inflation-reports/boxes-and-annexes-contained-in-inflation-reports/A-look-back-at-the-discontinued-exchange-rate-commitment)
was introduced on 7 November 2013 and ended on 6 April 2017 (known-at-month-start flag from
December 2013 through April 2017). Sweden floated before our sample;
[Riksbank history](https://www.riksbank.se/en-gb/monetary-policy/the-inflation-target/history-of-the-inflation-target/).
For Sweden and Romania no varying regime instrument is selected here: the placeholder is zero.
Zero placeholders do **not** mean a fixed exchange rate. Country regime definitions differ.

The instrument audit below checks variation and rank on full and common samples. Constant
regime columns must be excluded from later designs; a country's short regime segment should
also be assessed before use. This does not choose final instruments or estimate their coefficients.


In [5]:
datasets, input_panels, sample_rows, date_rows, checks = {}, {}, [], [], []
for country, info in COUNTRIES.items():
    local_level = levels[country].level.reindex(grid)
    local_fx = national_fx[country]
    inflation_local = national_hicp[country].pct_change(12, fill_method=None) * 100
    local_return = np.log(local_level).diff()  # Reindex first: no multi-month returns across gaps.
    d = hu[global_columns].reindex(grid).copy()
    d["r_local_lc"] = local_return
    d["fx_local_eur"] = -np.log(local_fx).diff()
    d["fx_local_usd"] = -np.log(local_fx / usd_per_eur).diff()
    for currency, cash in [("usd", "rf_usd"), ("eur", "rf_eur_proxy")]:
        d[f"r_local_{currency}"] = local_return + d[f"fx_local_{currency}"]
        d[f"rx_local_{currency}"] = d[f"r_local_{currency}"] - d[cash]
    gap_usd = inflation_us - inflation_local
    gap_eur = inflation_ea - inflation_local
    d["inflation_gap_usd_lag2_pp"] = gap_usd.shift(2)
    d["inflation_gap_eur_lag2_pp"] = gap_eur.shift(2)
    d["inflation_gap_change_usd_lag2_pp"] = gap_usd.diff().shift(2)
    d["inflation_gap_change_eur_lag2_pp"] = gap_eur.diff().shift(2)
    d["other_fx_gap_usd_lag2_pp"] = (inflation_us - inflation_ea).shift(2)
    d["other_fx_gap_eur_lag2_pp"] = -d["other_fx_gap_usd_lag2_pp"]
    d["regime_indicator"] = (grid >= pd.Period("2000-05", "M")).astype(int) if country == "PL" else (
        ((grid >= pd.Period("2013-12", "M")) & (grid < pd.Period("2017-05", "M"))).astype(int) if country == "CZ" else 0)
    d["flag_known_index_backcast"] = ((grid < pd.Period(info["known_backcast_before"], "M")).astype(int)
        if info["known_backcast_before"] else 0)
    d = d.loc[first:last]
    d.index.name = "month"
    complete = d.notna().all(axis=1)
    assert complete.any(), f"No complete {country} sample."
    beginning = d.index[complete][0]
    working = d.loc[beginning:last].copy()
    assert working.notna().all().all() and np.isfinite(working.to_numpy()).all(), f"Internal gaps or missing September endpoint: {country}"
    assert working.index.equals(pd.period_range(beginning, last, freq="M"))
    for market in ["local", "world", "us"]:
        np.testing.assert_allclose(working[f"r_{market}_usd"] - working[f"r_{market}_eur"], working.fx_eur_usd, atol=1e-12, rtol=0)
    np.testing.assert_allclose(working.fx_local_usd - working.fx_local_eur, working.fx_eur_usd, atol=1e-12, rtol=0)
    for currency, cash in [("usd", "rf_usd"), ("eur", "rf_eur_proxy")]:
        np.testing.assert_allclose(working[f"rx_local_{currency}"] + working[cash], working[f"r_local_{currency}"], atol=1e-12, rtol=0)
    np.testing.assert_array_equal(working[global_columns].to_numpy(), hu.loc[working.index, global_columns].to_numpy())
    end_dates = pd.DataFrame({"equity_date": levels[country].date, "fx_date": fx_dates[country]}).reindex(working.index)
    for name in end_dates:
        assert end_dates[name].notna().all()
        assert (end_dates.index.to_timestamp("M") - end_dates[name]).dt.days.between(0, 7).all(), f"Stale {country} {name}"
    end_dates["country"] = country
    date_rows.append(end_dates.reset_index(names="month"))
    datasets[country] = working
    input_panels[country] = pd.DataFrame({"index_level": local_level, "local_per_eur": local_fx,
        "usd_per_eur_inherited": usd_per_eur, "national_hicp": national_hicp[country],
        "us_inflation_raw_pp": inflation_us_raw, "us_inflation_for_z_pp": inflation_us,
        "national_inflation_pp": inflation_local, "ea_inflation_pp": inflation_ea}).loc[first:last]
    input_panels[country].index.name = "month"
    sample_rows.append({"country_code": country, **info, "first_return": str(beginning), "last_return": str(last),
        "months": len(working), "variables": working.shape[1], "internal_missing": int(working.isna().sum().sum()),
        "backcast_months_flagged": int(working.flag_known_index_backcast.sum()),
        "carried_US_inflation_months": int(working.flag_us_inflation_carried_lag2.sum())})
    checks.append({"country": country, "complete_endpoint": True, "contiguous": True,
        "conversion_identity": True, "excess_return_identity": True, "shared_inputs_exact": True,
        "month_end_age_within_7_days": True})

common_start = max(d.index[0] for d in datasets.values())
common_index = pd.period_range(common_start, last, freq="M", name="month")
assert hu.index.intersection(common_index).equals(common_index)
samples = pd.DataFrame(sample_rows).set_index("country_code")
samples["common_start"] = str(common_start)
samples["common_months"] = len(common_index)
display(samples[["country", "currency", "index", "first_return", "last_return", "months", "common_start", "common_months"]])


,country,currency,index,first_return,last_return,months,common_start,common_months
country_code,,,,,,,,
PL,Poland,PLN,WIG,2000-01,2026-09,321,2014-10,144
CZ,Czechia,CZK,PX-TR,2006-04,2026-09,246,2014-10,144
RO,Romania,RON,BET-TR,2014-10,2026-09,144,2014-10,144
SE,Sweden,SEK,OMXSGI,2008-12,2026-09,214,2014-10,144


## Export comparable investor views and audit instruments

Each country gets a wide `dataset.csv` containing both perspectives and simple, consistently named
`USD.csv` / `EUR.csv` model-input views. Currency factors are local appreciation against the investor
currency and EUR/USD (USD investors) or USD/EUR (EUR investors). The common sample also includes
the original Hungarian observations, renamed only for compatibility. Do not compare full-sample
estimates across countries as if they were estimated over the common window.

Inflation levels/changes are lagged **before** sample trimming. For common-model comparisons,
normalize instruments on that model's actual sample, rather than retaining Hungary's full-sample scaling.


In [6]:
global_z = ["short_rate_change_lag1_pp", "term_spread_change_lag1_pp", "credit_spread_lag1_pp"]
exports, instrument_rows, return_rows, common_views = {}, [], [], []
regime_definitions = {"PL": "Free float known from May 2000; only four earlier requested months",
    "CZ": "Exchange-rate floor known at month start: Dec 2013-Apr 2017",
    "RO": "Zero placeholder; no varying regime instrument selected", "SE": "Zero placeholder; float predates sample"}

def investor_view(d, currency):
    c = currency.lower()
    view = pd.DataFrame(index=d.index)
    for asset in ["world", "us", "local"]:
        view[f"r_{asset}"] = d[f"r_{asset}_{c}"]
        view[f"rx_{asset}"] = d[f"rx_{asset}_{c}"]
    view["rf"] = d["rf_usd" if currency == "USD" else "rf_eur_proxy"]
    view["fx_local"] = d[f"fx_local_{c}"]
    view["fx_other"] = d.fx_eur_usd * (1 if currency == "USD" else -1)
    view[global_z] = d[global_z]
    view["inflation_gap_lag2_pp"] = d[f"inflation_gap_{c}_lag2_pp"]
    view["inflation_gap_change_lag2_pp"] = d[f"inflation_gap_change_{c}_lag2_pp"]
    view["other_fx_gap_lag2_pp"] = d[f"other_fx_gap_{c}_lag2_pp"]
    view["regime_indicator"] = d.regime_indicator
    view["flag_us_inflation_carried_lag2"] = d.flag_us_inflation_carried_lag2
    view["flag_known_index_backcast"] = d.flag_known_index_backcast
    view.index.name = "month"
    return view

for country, d in datasets.items():
    exports[f"{country}/dataset.csv"] = d
    exports[f"{country}/monthly_inputs.csv"] = input_panels[country]
    for currency in ["USD", "EUR"]:
        view = investor_view(d, currency)
        exports[f"{country}/{currency}.csv"] = view
        matched = view.loc[common_index].copy()
        exports[f"common/{country}_{currency}.csv"] = matched
        common_views.append(matched.assign(country=country, perspective=currency).reset_index(names="month"))
        for sample, frame in [("available", view), ("common", matched)]:
            candidates = global_z + ["inflation_gap_change_lag2_pp", "other_fx_gap_lag2_pp", "regime_indicator"]
            varying = [name for name in candidates if frame[name].nunique() > 1]
            z = frame[varying]
            z = (z - z.mean()) / z.std(ddof=0)
            design = np.column_stack([np.ones(len(z)), z])
            rank = np.linalg.matrix_rank(design)
            assert rank == design.shape[1], f"Collinear candidate instruments: {country} {currency} {sample}"
            instrument_rows.append({"country": country, "perspective": currency, "sample": sample,
                "months": len(frame), "rank": rank, "columns_with_intercept": design.shape[1],
                "condition_number": np.linalg.cond(design),
                "constant_candidates": "; ".join(name for name in candidates if name not in varying),
                "regime_zeros": int(frame.regime_indicator.eq(0).sum()),
                "regime_ones": int(frame.regime_indicator.eq(1).sum()), "regime_definition": regime_definitions[country]})
            for name in ["rx_local", "fx_local", "fx_other"]:
                s = frame[name] * 100
                return_rows.append({"country": country, "perspective": currency, "sample": sample,
                    "series": name, "months": len(s), "mean_log_pp": s.mean(), "sd_log_pp": s.std(),
                    "min_log_pp": s.min(), "max_log_pp": s.max()})

# Preserve Hungarian observations exactly; derive only the extra lagged audit columns before trimming.
hu_common = hu.rename(columns={c: c.replace("_hu_", "_local_").replace("fx_huf_", "fx_local_") for c in hu})
hu_common["regime_indicator"] = hu.floating_regime_known
hu_common["inflation_gap_change_usd_lag2_pp"] = hu.inflation_gap_usd_lag2_pp.diff()
hu_common["inflation_gap_change_eur_lag2_pp"] = hu.inflation_gap_eur_lag2_pp.diff()
hu_common["other_fx_gap_usd_lag2_pp"] = (inflation_us - inflation_ea).shift(2).reindex(hu.index)
hu_common["other_fx_gap_eur_lag2_pp"] = -hu_common.other_fx_gap_usd_lag2_pp
hu_common["flag_known_index_backcast"] = 0
for currency in ["USD", "EUR"]:
    matched = investor_view(hu_common, currency).loc[common_index]
    assert matched.notna().all().all()
    exports[f"common/HU_{currency}.csv"] = matched
    common_views.append(matched.assign(country="HU", perspective=currency).reset_index(names="month"))

exports["sample_summary.csv"] = samples
source_rows, coverage_rows = [], []
source_names = {"PL": "poland_wig.json", "CZ": "czech_pxtr.csv", "RO": ro_names[0], "SE": "sweden_omxsgi.csv"}
for country, info in COUNTRIES.items():
    source_rows.append({"country": country, "index": info["index"], "currency": info["currency"],
        "source_url": manifest["sources"][source_names[country]]["url"],
        "first_downloaded_numeric_level": str(levels[country].index.min()),
        "last_kept_level": str(levels[country].index.max()),
        "return_convention": "total return; dividends included", "ETF_substitution": False,
        "price_only_splicing": False, "download_vintage": SNAPSHOT_DATE,
        "known_backcast_before": info["known_backcast_before"] or "none identified in used coverage"})
    for variable, series in input_panels[country].items():
        used_sample = series.reindex(datasets[country].index)
        coverage_rows.append({"country": country, "input": variable,
            "first_available_requested": str(series.first_valid_index()),
            "last_available_requested": str(series.last_valid_index()),
            "missing_requested": int(series.isna().sum()), "missing_working": int(used_sample.isna().sum())})
exports["source_audit.csv"] = pd.DataFrame(source_rows)
exports["input_coverage.csv"] = pd.DataFrame(coverage_rows)
exports["instrument_audit.csv"] = pd.DataFrame(instrument_rows)
exports["return_summary.csv"] = pd.DataFrame(return_rows)
exports["quality_checks.csv"] = pd.DataFrame(checks)
exports["month_end_dates.csv"] = pd.concat(date_rows, ignore_index=True)
exports["common_panel.csv"] = pd.concat(common_views, ignore_index=True)

for name, frame in exports.items():
    path = RUN / name
    content = frame.to_csv(index=name not in ["source_audit.csv", "input_coverage.csv", "instrument_audit.csv", "return_summary.csv", "quality_checks.csv", "month_end_dates.csv", "common_panel.csv"], float_format="%.17g", lineterminator="\n").encode()
    if manifest["sealed"]:
        assert name in manifest["exports"] and digest(content) == manifest["exports"][name], f"Changed frozen export: {name}"
        assert path.exists() and digest(path.read_bytes()) == manifest["exports"][name], f"Missing/corrupt frozen export: {name}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
        manifest["exports"][name] = digest(content)

if not manifest["sealed"]:
    manifest.update({"sealed": True, "countries": COUNTRIES, "regime_definitions": regime_definitions,
        "common_start": str(common_start), "common_end": str(last), "common_months": len(common_index),
        "python": sys.executable, "environment": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "requests", "ipykernel"]},
        "notes": ["Unhedged monthly log returns in decimals; predictor units are annual percentage points",
                  "National vintage 2026-11-16; common global inputs inherited from Hungarian vintage 2026-11-13",
                  "No interpolated returns/FX, price-only splicing, ETF substitution or personal credentials"]})
    save_manifest()

# Small portable lock for subsequent notebooks; all raw data and exports remain local.
registry = {"version": 1, "root": RUN.relative_to(ROOT).as_posix(), "settings": settings,
    "manifest_sha256": digest(MANIFEST.read_bytes()), "common_start": str(common_start), "common_end": str(last),
    "common_months": len(common_index), "files": {**{(RAW / name).relative_to(ROOT).as_posix(): r["sha256"] for name, r in manifest["sources"].items()},
        **{(RUN / name).relative_to(ROOT).as_posix(): sha for name, sha in manifest["exports"].items()}, **manifest["inherited"]}}
registry_path = ROOT / "regional_dataset_inputs.json"
registry_content = json.dumps(registry, indent=2, sort_keys=True) + "\n"
if registry_path.exists():
    assert registry_path.read_text() == registry_content, "Regional input lock changed; inspect before replacing it."
else: registry_path.write_text(registry_content)
print("Sealed and verified", len(manifest["sources"]), "raw sources and", len(exports), "exports.")


Sealed and verified 174 raw sources and 34 exports.


## Final summary

The **available** samples retain each country's history; the **common** files have identical months
for all four countries and Hungary. Returns and FX statistics below are monthly log percentage
points, not annualized. `instrument_audit.csv` reports constant candidates and short regime segments.
Known back-calculated months are retained and flagged for later robustness checks.

Re-running this notebook after sealing is offline and verifies all source/export checksums.
Missing/corrupt files cause an error; a refresh never silently replaces the fixed vintage.
Keep the complete snapshot directory and the original Hungarian snapshot. For a revised national
vintage, use a separate snapshot date and preserve the existing input lock.


The inherited annual-inflation carry flag marks December 2025 after lagging. The USD local-gap **change** predictor is also affected in January 2026; the EUR local-gap change is unaffected by US CPI. The other-FX gap **level** is affected in December 2025 for both perspectives. Later carry-sensitivity tests must follow these transformations rather than treating the level flag as a change-predictor flag.


In [7]:
display(samples[["country", "currency", "index", "first_return", "last_return", "months", "common_start", "common_months", "backcast_months_flagged", "internal_missing"]])
display(pd.DataFrame(source_rows)[["country", "index", "first_downloaded_numeric_level", "last_kept_level", "ETF_substitution", "price_only_splicing"]])
display(pd.DataFrame(instrument_rows)[["country", "perspective", "sample", "months", "rank", "columns_with_intercept", "constant_candidates", "regime_zeros", "regime_ones"]])
display(pd.DataFrame(return_rows).query("series == 'rx_local'").round(4))
display(Markdown(f"**Frozen endpoint:** {END_MONTH}. **Common window:** {common_start}–{last}, **{len(common_index)} months**. Snapshot: `{RUN.relative_to(ROOT)}`. Input lock: `regional_dataset_inputs.json`."))


,country,currency,index,first_return,last_return,months,common_start,common_months,backcast_months_flagged,internal_missing
country_code,,,,,,,,,,
PL,Poland,PLN,WIG,2000-01,2026-09,321,2014-10,144,0,0
CZ,Czechia,CZK,PX-TR,2006-04,2026-09,246,2014-10,144,96,0
RO,Romania,RON,BET-TR,2014-10,2026-09,144,2014-10,144,0,0
SE,Sweden,SEK,OMXSGI,2008-12,2026-09,214,2014-10,144,0,0


,country,index,first_downloaded_numeric_level,last_kept_level,ETF_substitution,price_only_splicing
0,PL,WIG,1999-12,2026-09,False,False
1,CZ,PX-TR,2006-03,2026-09,False,False
2,RO,BET-TR,2014-09,2026-09,False,False
3,SE,OMXSGI,2008-11,2026-09,False,False


,country,perspective,sample,months,rank,columns_with_intercept,constant_candidates,regime_zeros,regime_ones
0,PL,USD,available,321,7,7,,4,317
1,PL,USD,common,144,6,6,regime_indicator,0,144
2,PL,EUR,available,321,7,7,,4,317
3,PL,EUR,common,144,6,6,regime_indicator,0,144
4,CZ,USD,available,246,7,7,,205,41
5,CZ,USD,common,144,7,7,,113,31
6,CZ,EUR,available,246,7,7,,205,41
7,CZ,EUR,common,144,7,7,,113,31
8,RO,USD,available,144,6,6,regime_indicator,144,0
9,RO,USD,common,144,6,6,regime_indicator,144,0


,country,perspective,sample,series,months,mean_log_pp,sd_log_pp,min_log_pp,max_log_pp
0,PL,USD,available,rx_local,321,0.5382,8.3876,-44.6885,24.9464
3,PL,USD,common,rx_local,144,0.4656,7.1915,-22.2509,23.4031
6,PL,EUR,available,rx_local,321,0.5292,7.2826,-33.5700,25.0990
9,PL,EUR,common,rx_local,144,0.6459,6.0592,-21.8988,21.0759
12,CZ,USD,available,rx_local,246,0.5552,7.3716,-41.3613,23.2992
15,CZ,USD,common,rx_local,144,1.0032,5.7512,-28.8797,20.0106
18,CZ,EUR,available,rx_local,246,0.6277,6.0657,-30.2429,18.9011
21,CZ,EUR,common,rx_local,144,1.1835,4.9100,-28.5275,17.6834
24,RO,USD,available,rx_local,144,1.2008,5.5668,-18.5295,12.5787
27,RO,USD,common,rx_local,144,1.2008,5.5668,-18.5295,12.5787


**Frozen endpoint:** 2026-09. **Common window:** 2014-10–2026-09, **144 months**. Snapshot: `data/regional_fixed_2026-11-16_end_2026-09/from_2000-01`. Input lock: `regional_dataset_inputs.json`.